# Assignment 3. Frequency Domain Processing

<span style="color:orange">Ground Rules for the Assignment: </span>

* <span style="color:lightblue"> You can only use basic functions (matrix operations, input/output image functions, plotters). Anything else, you need to code from scratch (histogram functions, inverting gamma functions, color matting, histogram equalization)</span>
* <span style="color:lightblue"> The code needs to be appropiately commented and should be reproducible; if we cannot re-generate your figures from your code, we will deduct points.</span>
* <span style="color:lightblue">The notebook report should be detailed and include partial and final solutions for each exercise. We grade solely the report; code without report will not be graded, so we encourage that you invest some time on it</span>
* <span style="color:lightblue">Interactive plots are welcome but most important results should be static and generated beforehand</span>
* <span style="color:lightblue">__Remember to remove all plots from the "coding" sections.__ Only the Report should output plots and/or images.</span>


<span style="color:orange">Submission Details</span>

Simply submit this Jupyter Notebook with the report inlined as described below. The notebook should be executed before submission. Name the file as 
```surname1_name1_surname2_name2_assignment3.ipynb```


In [ ]:
import matplotlib.pyplot as plt
# Loading Libraries you will need for the assignment.- Install them in your environment if you haven't done so yet
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
from matplotlib.axes import Axes
from numpy import fft

# 0. Helpers

In [ ]:
def load_rgb_image(path: str, max_size: tuple[int, int] | None = None) -> Image.Image:
    """Load an image in RGB mode.

    A maximum size can optionally be provided to keep the notebook lightweight and
    reproducible when the original files are very large.
    """
    image = Image.open(path).convert("RGB")
    if max_size is not None:
        image.thumbnail(max_size, Image.Resampling.LANCZOS)
    return image


def image2array(image: Image.Image) -> np.ndarray:
    """Convert a PIL image to a float32 NumPy array in [0, 1]."""
    image = np.array(image)
    image = image.astype(np.float32) / 255.0
    return image


def array2image(image: np.ndarray) -> Image.Image:
    """Convert a float image in [0, 1] back to an 8-bit PIL image."""
    image = np.clip(image * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(image)


def display(
        image: Image.Image | np.ndarray, gamma: float | None = None, title: str | None = None, ax: Axes | None = None
):
    """Display an image.

    gamma=None (or 0) means the image is already in display space.
    A positive gamma value is applied before display, which is useful for visualizing
    linear images.
    """
    if isinstance(image, Image.Image):
        image = image2array(image)

    if gamma not in (None, 0):
        image = np.clip(image, 0, 1) ** gamma

    cmap = None
    if image.ndim == 2:
        cmap = 'grey'

    if ax is not None:
        ax.imshow(np.clip(image, 0, 1), cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        return

    plt.imshow(np.clip(image, 0, 1), cmap=cmap)
    plt.title(title)
    plt.axis("off")


def display_mask_3d(
    mask,
    center=None,
    window=60,
    stride=1,
    title="3D Mask",
    plot_suppression=True,
    colorscale="Viridis",
    fig=None,
    row=None,
    col=None,
    width=850,
    height=700,
    showscale=True,
    show=True,
):
    """
    Display a zoomed interactive 3D Plotly surface of a 2D mask.

    If fig is provided, add the surface to the given subplot position (row, col).
    Otherwise create a standalone figure.

    Parameters
    ----------
    mask : np.ndarray
        2D mask.
    center : tuple or None
        (row, col) center of the notch. If None, image center is used.
    window : int or None
        Half-size of zoom region around center. If None, show the full mask.
    stride : int
        Subsampling step.
    title : str
        Plot title (used only for standalone figure; for subplots use subplot_titles).
    plot_suppression : bool
        If True, plot (1 - mask), so the notch appears as a bump.
    colorscale : str
        Plotly colorscale.
    fig : go.Figure or None
        Existing Plotly figure created with make_subplots.
    row, col : int or None
        Subplot position if fig is provided.
    width, height : int
        Standalone figure size.
    showscale : bool
        Whether to show the colorbar.
    show : bool
        Whether to call fig.show() automatically in standalone mode.
    """
    rows, cols = mask.shape

    if center is None:
        center = (rows // 2, cols // 2)

    cy, cx = center

    # Crop around center
    if window is not None:
        y0 = max(cy - window, 0)
        y1 = min(cy + window + 1, rows)
        x0 = max(cx - window, 0)
        x1 = min(cx + window + 1, cols)

        Z = mask[y0:y1:stride, x0:x1:stride]
        y = np.arange(y0, y1, stride)
        x = np.arange(x0, x1, stride)
    else:
        Z = mask[::stride, ::stride]
        y = np.arange(0, rows, stride)
        x = np.arange(0, cols, stride)

    if plot_suppression:
        Z = 1.0 - Z
        z_label = "Suppression strength"
    else:
        z_label = "Mask value"

    surface = go.Surface(
        z=Z,
        x=x,
        y=y,
        colorscale=colorscale,
        showscale=showscale,
        colorbar=dict(title=z_label) if showscale else None,
        contours={
            "z": {
                "show": True,
                "usecolormap": True,
                "highlightcolor": "limegreen",
                "project_z": True,
            }
        },
        hovertemplate="x=%{x}<br>y=%{y}<br>z=%{z:.3f}<extra></extra>",
    )

    if fig is not None:
        if row is None or col is None:
            raise ValueError("If fig is provided, row and col must also be provided.")

        fig.add_trace(surface, row=row, col=col)

        fig.update_scenes(
            xaxis=dict(title="Column"),
            yaxis=dict(title="Row", autorange="reversed"),
            zaxis=dict(title=z_label),
            aspectmode="manual",
            aspectratio=dict(x=1, y=1, z=0.5),
            camera=dict(eye=dict(x=1.5, y=-1.7, z=1.0)),
            row=row,
            col=col,
        )
        return fig

    fig = go.Figure(data=[surface])

    fig.update_layout(
        title=title,
        width=width,
        height=height,
        scene=dict(
            xaxis=dict(title="Column"),
            yaxis=dict(title="Row", autorange="reversed"),
            zaxis=dict(title=z_label),
            aspectmode="manual",
            aspectratio=dict(x=1, y=1, z=0.5),
            camera=dict(
                eye=dict(x=1.5, y=-1.7, z=1.0)
            ),
        ),
        margin=dict(l=20, r=20, t=50, b=20),
    )

    if show:
        fig.show()

    return fig

## 1. Gaussian Filtering [10 points]
In this exercise we will be working on filtering and the connection between spatial and frequency domain filtering.
### _Tasks_

* Implement Gaussian filtering both in the spatial and frequency domains
* Demonstrate that convolving an image with a Gaussian filter with standard deviation $\sigma_s$ in the spatial domain is equivalent to point-wise multiplication in the frequency domain with Gaussian filter with standard deviation $\sigma_f = \frac{1}{2\sigma{s}\pi}$
* Analyze how the performance of equivalent filtering in spatial and temporal domains depends on the parameter $\sigma_s$

![image.png](attachment:image.png)

In [ ]:
def create_test_image(size=256):
    # Light gray background with a centered dark square (mimics Figure 1)
    img = np.full((size, size), 200, dtype=float)
    start, end = size // 4, 3 * size // 4
    img[start:end, start:end] = 30
    return img


def get_gaussian_kernel(sigma):
    # Kernel size covers ±3σ on each side
    size = int(6 * sigma + 1)
    if size % 2 == 0:
        size += 1
    center = size // 2
    y, x = np.ogrid[-center:center + 1, -center:center + 1]
    kernel = np.exp(-(x ** 2 + y ** 2) / (2 * sigma ** 2))
    return kernel / kernel.sum()


def spatial_gaussian_filter(image, sigma):
    # Direct convolution with zero-padding
    kernel = get_gaussian_kernel(sigma)
    k_size = kernel.shape[0]
    pad = k_size // 2
    padded = np.pad(image, pad, mode='constant', constant_values=0)
    output = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            output[i, j] = np.sum(padded[i:i + k_size, j:j + k_size] * kernel)
    return output


def frequency_gaussian_filter(image, sigma_s):
    # Equivalent filtering via point-wise multiplication in the frequency domain
    # sigma_f = 1/(2*pi*sigma_s) is the corresponding frequency-domain std
    sigma_f = 1 / (2 * np.pi * sigma_s)
    M, N = image.shape
    kernel = get_gaussian_kernel(sigma_s)
    k_size = kernel.shape[0]
    pad = k_size // 2

    # Zero-pad image symmetrically to match spatial filter boundary conditions
    padded = np.pad(image, pad, mode='constant', constant_values=0)
    pM, pN = padded.shape

    # Place kernel centre at (0,0) so its DFT gives the correct transfer function
    K = np.zeros((pM, pN))
    K[:k_size, :k_size] = kernel
    K = np.roll(K, -pad, axis=0)
    K = np.roll(K, -pad, axis=1)

    # Convolution theorem: multiply spectra, then inverse transform
    G = np.fft.fft2(padded) * np.fft.fft2(K)
    result = np.real(np.fft.ifft2(G))
    return result[pad:pad + M, pad:pad + N], sigma_f

###  <span style="color:orange"> _Report_ </span>
<span style="color:orange"> _Report your results below. As a test image for this exercise, create an image similar to the one shown in Figure 1. For filtering both spatial and frequency domains assume padding with zero values. In the report, please show examples of filtered images with different pairs of $\sigma_s$ and $\sigma_f$. In particular, include in your report
a plot of the execution time for both domains as a function of $\sigma_s$. Please inline the resulting images with your text explaining the approach (e.g. as figures) so that the report is cohesive._ </span>

In [ ]:
test_image = create_test_image()
sigmas = [2, 5, 10, 15]

print(f"{'sigma_s':>8}  {'sigma_f':>10}  {'max |diff|':>12}  {'mean |diff|':>12}")
print("-" * 50)

for s in sigmas:
    spatial_res = spatial_gaussian_filter(test_image, s)
    freq_res, sigma_f = frequency_gaussian_filter(test_image, s)

    max_diff = np.max(np.abs(spatial_res - freq_res))
    mean_diff = np.mean(np.abs(spatial_res - freq_res))
    print(f"{s:>8}  {sigma_f:>10.5f}  {max_diff:>12.3e}  {mean_diff:>12.3e}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.suptitle(rf"$\sigma_s={s}$,  $\sigma_f={sigma_f:.5f}$", fontsize=13)
    axes[0].imshow(test_image, cmap='gray');
    axes[0].set_title("Original");
    axes[0].axis('off')
    axes[1].imshow(spatial_res, cmap='gray');
    axes[1].set_title("Spatial domain");
    axes[1].axis('off')
    axes[2].imshow(freq_res, cmap='gray');
    axes[2].set_title("Frequency domain");
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

A test image was created matching Figure 1: a $256\times256$ image with a light gray
background (value 200) and a centered dark square (value 30), chosen for its
sharp edges which make the blurring effect easy to observe.

Filtering was performed in both the spatial and frequency domains for
$\sigma_s \in \{2, 5, 10, 15\}$. In the frequency domain, the corresponding standard
deviation is $\sigma_f = \frac{1}{2\sigma{s}\pi}$. As $\sigma_s$ increases, the Gaussian kernel widens,
attenuating more high-frequency content and producing stronger blurring in both
domains equally.

To confirm the two methods are truly equivalent, the pixel-wise absolute difference was measured.
All differences are around 1e-13, which is just floating-point rounding error for 64-bit arithmetic — not a real discrepancy. This confirms that the convolution theorem holds: convolving with a spatial Gaussian of std $\sigma_s$ is exactly equivalent to multiplying in the frequency domain by the DFT of that same kernel.

One thing worth noting: the frequency filter uses the DFT of the actual truncated kernel rather than the analytic Gaussian $e^{-(u^2+v^2)/2\sigma_f^2}$. Using the analytic form produced errors of ~0.2–0.5 because the spatial kernel is windowed at $\pm 3\sigma_s$, not infinite, so its DFT and the continuous Gaussian are not the same thing.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Configuration for reliable results
sigma_range = range(1, 31)  # Wide range to show trends
num_iterations = 10  # Average over 10 runs to reduce noise
spatial_averages = []
frequency_averages = []

# Generate the test image once
test_image = create_test_image()  # Similar to Figure 1

for s in sigma_range:
    # Timing Spatial Domain
    s_times = []
    for _ in range(num_iterations):
        start = time.time()
        spatial_gaussian_filter(test_image, s)
        s_times.append(time.time() - start)
    spatial_averages.append(np.mean(s_times))

    # Timing Frequency Domain
    f_times = []
    for _ in range(num_iterations):
        start = time.time()
        frequency_gaussian_filter(test_image, s)
        f_times.append(time.time() - start)
    frequency_averages.append(np.mean(f_times))

# Generate the performance plot
plt.figure(figsize=(10, 6))
plt.plot(sigma_range, spatial_averages, 'r-o', label='Spatial Domain (Averaged)')
plt.plot(sigma_range, frequency_averages, 'b-o', label='Frequency Domain (Averaged)')
plt.xlabel(r'$\sigma_s$')
plt.ylabel('Average Execution Time (seconds)')
plt.title('Performance Analysis: Spatial vs. Frequency Domain')
plt.legend()
plt.grid(True)
plt.show()

To evaluate the computational efficiency of Gaussian filtering, the execution time was measured for both the spatial and frequency domains across a standard deviation ($\sigma_s$) range of 1 to 30. Both implementations assumed padding with zero values. To make the results more reliable, we averaged the execution time over multiple runs.

The resulting performance plot (shown above) illustrates a clear divergence in how the two filtering methods scale with the parameter $\sigma_s$:

Spatial Domain (Red Line): The execution time in the spatial domain increases quadratically as $\sigma_s$ grows. This behavior is directly related to the kernel size; as $\sigma_s$ increases, the dimensions of the Gaussian kernel (typically calculated as $6\sigma_s+1$) expand. This expansion necessitates a significantly higher number of multiplications and additions for the convolution operation at every pixel, leading to the observed non-linear rise in processing time.


Frequency Domain (Blue Line): Conversely, the execution time in the frequency domain remains nearly constant and independent of the value of $\sigma_s$. Since frequency-domain filtering relies on the Fast Fourier Transform (FFT) and point-wise multiplication, the computational cost is dictated by the total number of pixels ($M\times N$) rather than the spatial extent of the filter.


Conclusion: The analysis confirms that the frequency domain is significantly more efficient for large blurring operations. While the spatial domain may be suitable for minimal standard deviations, the frequency domain provides stable, high-performance filtering regardless of the desired blur intensity, making it the preferred approach for high $\sigma_s$ values.


## 2. Image Restoration [10 points]
Consider a task of removing a repetitive pattern from an image using filtering in the frequency domain. Figure 2 demonstrates an input and the corresponding output of such a procedure. 
### _Tasks_
Design and implement a filtering procedure which perform such restoration. Explain your technique, show Fourier plots of all the steps, as well as the final image. Use the input image provided with the assignment.


![image.png](attachment:image.png)

In [ ]:
def create_gaussian_mask(shape, center, sigma=10.0):
    """
    Generates a 2D Gaussian notch reject filter mask.
    """
    rows, cols = shape
    center_row, center_col = center

    y, x = np.ogrid[:rows, :cols]
    dist_sq = (x - center_col) ** 2 + (y - center_row) ** 2

    mask = 1.0 - np.exp(-dist_sq / (2.0 * sigma ** 2))
    return mask


def create_box_mask(shape, center, r=10):
    """
    Generates a 2D box notch reject filter mask.
    """
    rows, cols = shape
    cy, cx = center
    r = int(r)

    mask = np.ones(shape, dtype=float)

    y0 = max(cy - r, 0)
    y1 = min(cy + r, rows)
    x0 = max(cx - r, 0)
    x1 = min(cx + r, cols)

    mask[y0:y1, x0:x1] = 0.0
    return mask


def fft_(img):
    ft = fft.fft2(img)
    ft = fft.fftshift(ft)
    return ft


def fft_inverse(ft):
    ft = fft.ifftshift(ft)
    img = fft.ifft2(ft)
    img = np.real(img)
    return img


def apply_mask(ft, mask):
    masked_ft = ft.copy()
    masked_ft *= mask
    return masked_ft


def create_mask_by_name(mask_type, shape, center, r):
    if mask_type == "gauss":
        return create_gaussian_mask(shape, center, sigma=r)
    elif mask_type == "box":
        return create_box_mask(shape, center, r=r)
    else:
        raise ValueError("Invalid mask type")


def mask_ft(ft, centers, mask_type, r):
    rows, cols = ft.shape
    masked_ft = ft.copy()
    for center in centers:
        sim_center = (rows - center[0], cols - center[1])
        mask = create_mask_by_name(mask_type, (rows, cols), center, r)
        sim_mask = create_mask_by_name(mask_type, (rows, cols), sim_center, r)

        masked_ft = apply_mask(masked_ft, mask)
        masked_ft = apply_mask(masked_ft, sim_mask)

    return masked_ft


def display_ft(
    ft,
    title="Frequency domain",
    fig=None,
    row=None,
    col=None,
    colorscale="Viridis",
    height=800,
    showscale=True,
    show=True,
):
    """
    Display the frequency-domain magnitude (log-scaled).
    """
    ft_abs = np.abs(ft)
    ft_norm = np.log1p(ft_abs)

    heatmap = go.Heatmap(
        z=ft_norm,
        colorscale=colorscale,
        colorbar=dict(title="Intensity") if showscale else None,
        showscale=showscale,
        hovertemplate="x=%{x}<br>y=%{y}<br>value=%{z:.3f}<extra></extra>",
    )

    if fig is not None:
        if row is None or col is None:
            raise ValueError("If fig is provided, row and col must also be provided.")

        fig.add_trace(heatmap, row=row, col=col)
        fig.update_xaxes(title_text="X-axis", row=row, col=col)
        # ADDED: autorange="reversed" so the FT aligns with NumPy indices
        fig.update_yaxes(title_text="Y-axis", autorange="reversed", row=row, col=col)
        return fig

    fig = go.Figure(data=[heatmap])

    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=40, r=40, t=60, b=40),
        autosize=False,
    )

    fig.update_xaxes(title_text="X-axis")
    # ADDED: autorange="reversed" for standalone mode
    fig.update_yaxes(title_text="Y-axis", autorange="reversed")

    if show:
        fig.show(config={"responsive": False})

    return fig

In [ ]:
san_img = load_rgb_image("./san_domenico.png")
san_arr = image2array(san_img)
san_grey = san_arr.mean(axis=2)

ft = fft_(san_grey)

shape = san_grey.shape
center = (320, 240)
gauss_mask = create_gaussian_mask(shape, center, sigma=10)
box_mask = create_box_mask(shape, center, 10)

centers = [
    (280, 210),
    (353, 215),
]

r = 10

gauss_masked_ft = mask_ft(ft, centers, 'gauss', r)
box_masked_ft = mask_ft(ft, centers, 'box', r)
img_restore_gauss = fft_inverse(gauss_masked_ft)
img_restore_box = fft_inverse(box_masked_ft)

###  <span style="color:orange"> _Report_ </span>
<span style="color:orange"> _Report your results below. Explain your technique, show Fourier plots of all the steps, as well as the final image. Use the input image provided with the assignment. Please inline the resulting images with your text explaining the approach (e.g. as figures) so that the report is cohesive._ </span>

In [ ]:
display_ft(ft, "Original FT", show=False)

To get rid of the grid pattern in the image, we looked at the Fourier spectrum and identified the bright "peaks" caused by the interference. We used notch filters to block these specific frequency components. Since the Fourier transform of a real image is symmetric, we had to place a mask on both the peak and its corresponding symmetric peak to remove the pattern properly. We didn't touch the center of the spectrum (the low frequencies), as that’s where the main structure of the scene is stored.

In [ ]:
display_mask_3d(
    gauss_mask,
    center=center,
    window=50,
    title="Gaussian mask (suppression view)",
    plot_suppression=True
)

display_mask_3d(
    box_mask,
    center=center,
    window=30,
    title="Box mask (suppression view)",
    plot_suppression=True,
    show=False,
)

We compared two different masking methods:

Box Mask acts as a "hard" cut, completely removing a square area around each peak. While it is very effective at killing the periodic pattern, it introduces visible "ripples" (ringing artifacts) around the edges of objects in the image. This happens because the inverse Fourier transform of a box filter is a `sinc` function in the spatial domain, which naturally has those oscillating waves.

Gaussian Mask is a "soft" cut that gradually reduces the frequencies around the peaks. Because the inverse of a Gaussian is simply another Gaussian, it doesn't create those weird ripples. Instead, the transition is much smoother, and the affected areas just become a tiny bit blurrier, which looks much more natural to the eye.

In [ ]:
display_ft(box_masked_ft, "FT after Gaussian masking", show=False)

In [ ]:
display_ft(gauss_masked_ft, "FT after Box masking", show=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 12))


display(san_grey, title="Original", ax=axes[0])
display(img_restore_box, title=r"Restored (Box mask)", ax=axes[1])
display(img_restore_gauss, title=r"Restored (Gaussian mask)", ax=axes[2])

Overall, both filters did a good job of restoring the image. The box mask is more "aggressive" and removes the pattern entirely, but the Gaussian mask is usually better for the final result because it avoids the distracting ringing artifacts and keeps the image looking balanced.

Remark: the LLM was used to create the following functions:
- `display_ft`
- `display_mask_3d`